<a href="https://colab.research.google.com/github/silvestrecarolina-star/python/blob/main/Assignment_3_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

path = ('/content/drive/MyDrive/kiva_loans.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
loans = pd.read_csv(path, parse_dates=['posted_time', 'disbursed_time', 'funded_time', 'date'])

# Question 1: Among Agriculture-sector loans posted in January to June, which activities have the most loans, and how many loans is each based on?

In [3]:
# create month, then select, group, sort
loans['month'] = loans['posted_time'].apply(lambda t: t.month)

by_activity = (loans[(loans['sector'] == 'Agriculture') & (loans['month'].between(1, 6))]
               .groupby('activity')
               .agg(n_loans=('id', 'size'))
               .sort_values('n_loans', ascending=False)
               .reset_index())
by_activity.head(10)

,activity,n_loans
0,Farming,43571
1,Agriculture,15866
2,Pigs,15179
3,Livestock,7600
4,Animal Sales,5491
5,Poultry,5335
6,Dairy,4736
7,Cattle,4677
8,Farm Supplies,2618
9,Flowers,437


**Answer.** In the first six months of the year (January to June), Farming had the most Agriculture (sector) loans by a wide margin, with 43 571 loans, nearly three times the next activity, Agriculture (activity), at 15 866. Pigs follows closely with 15 179, and activities such as Flowers have only a few hundred loans. These counts pool every year in the dataset, so they show which activities are most common overall, not a single year's pattern.

# Question 2: Among Pharmacy and Medical Clinic loans, using my own short, medium and long term classes (term_in_months), how many loans of each activity fall in each class, and what is the median loan_amount in each?



In [4]:
pharm = loans[loans['activity'].isin(['Pharmacy', 'Medical Clinic'])].copy()

def term_class(x):
  if x < 9:
    return 'short'
  elif x < 15:
    return 'medium'
  else:
    return 'long'

pharm['term_class'] = pharm['term_in_months'].apply(term_class)
by_term = (pharm.groupby(['activity', 'term_class'])
           .agg(n_loans=('id', 'size'), median_amount=('loan_amount', 'median'))
           .sort_values(['activity', 'median_amount'], ascending=[True, False])
           .reset_index())
by_term.round(1)

,activity,term_class,n_loans,median_amount
0,Medical Clinic,long,106,1325.0
1,Medical Clinic,medium,158,600.0
2,Medical Clinic,short,59,600.0
3,Pharmacy,long,158,1100.0
4,Pharmacy,short,383,950.0
5,Pharmacy,medium,471,750.0


**Answer.** In both activities the long-term loans (15 months or more) have the highest median loan_amount: 1 325 USD for Medical Clinic (106 loans) and 1 100 USD for Pharmacy (158 loans). Medical Clinic is ahead only in that long class. In the short and medium classes Pharmacy has the higher median, 950 and 750 USD against 600 USD for both Medical Clinic classes. Pharmacy's 471 medium loans have a lower median than its 383 short loans, so a longer term does not always mean a bigger loan. The smallest group is Medical Clinic short, with 59 loans.

# Question 3: Among Aquaculture loans, which countries have the longest median term_in_months, what is their median loan_amount, and how many loans is each based on?

In [5]:
by_country = (loans[(loans['activity'] == 'Aquaculture')]
        .groupby('country')
        .agg(n_loans=('id', 'size'), median_term=('term_in_months', 'median'), median_amount=('loan_amount', 'median'))
        .sort_values('median_term', ascending=False)
        .reset_index())
by_country[by_country['n_loans'] >=5]

,country,n_loans,median_term,median_amount
1,Cambodia,9,19.0,875.0
2,Colombia,14,18.5,612.5
5,El Salvador,5,14.0,500.0
8,Vietnam,30,13.0,1375.0
10,Philippines,40,8.0,300.0


**Answer.** Among countries with at least 5 Aquaculture loans, Cambodia has the longest median term at 19 months (9 loans, median 875 USD), just ahead of Colombia at 18.5 months (14 loans, median 612.5 USD). The Philippines has the shortest term, 8 months, and the smallest loans, 300 USD, but the largest sample at 40 loans. Longer terms do not go with bigger loans, since Vietnam has the largest median loan, 1 375 USD, with a mid-length term. Seven countries were left out because they had fewer than 5 loans each, so these medians describe only the five biggest Aquaculture markets.

# Question 4: Among Health loans whose use mentions "water", in which countries with at least 5 loans is the median loan_amount highest, how many lenders does a typical loan attract (mean lender_count), and how many loans is each based on?

In [6]:
water_by_country = (loans[(loans['sector'] == 'Health') & (loans['use'].str.contains('water', case=False, na=False))]
                    .groupby('country')
                    .agg(n_loans=('id', 'size'),
                         median_amount=('loan_amount', 'median'),
                         mean_lenders=('lender_count', 'mean'))
                    .sort_values('median_amount', ascending=False)
                    .reset_index())
water_by_country[water_by_country['n_loans'] >= 5].round(1)

,country,n_loans,median_amount,mean_lenders
0,Kenya,352,1800.0,52.6
2,Uganda,13,1025.0,36.0
4,Lao People's Democratic Republic,18,662.5,29.7
6,Indonesia,5,525.0,33.2
8,Cambodia,19,150.0,4.8
10,Vietnam,18,75.0,2.2


**Answer.** Among the six countries with at least 5 water-related Health loans, Kenya has the highest median loan, 1 800 USD, and its loans attract about 53 lenders each, based on 352 loans. Uganda follows at 1 025 USD (13 loans), while Cambodia and Vietnam have far smaller loans, 150 and 75 USD, with only 5 and 2 lenders per loan. Larger loans generally attract more lenders, but Kenya makes up 352 of the 432 water loans, so the other countries rest on only 5 to 19 loans each, and five countries with fewer than 5 loans were left out.

# Question 5: Among animal and crop loans in Agriculture, which type has the higher share of under-funded loans (funded_amount below loan_amount), and how many loans is each based on?

In [7]:
# create farm_type and under_funded, then select, group, sort
def farm_type(a):
    if a in ['Pigs', 'Poultry', 'Cattle', 'Dairy', 'Livestock', 'Animal Sales', 'Aquaculture', 'Beekeeping']:
        return 'animals'
    elif a in ['Farming', 'Agriculture', 'Flowers']:
        return 'crops'
    return 'other'

agri = loans[loans['sector'] == 'Agriculture'].copy()
agri['farm_type'] = agri['activity'].apply(farm_type)
agri['under_funded'] = agri['funded_amount'] < agri['loan_amount']

by_farm_type = (agri[agri['farm_type'].isin(['animals', 'crops'])]
                .groupby('farm_type')
                .agg(n_loans=('id', 'size'),
                     share_underfunded=('under_funded', 'mean'))
                .sort_values('share_underfunded', ascending=False)
                .reset_index())
by_farm_type.round(3)

,farm_type,n_loans,share_underfunded
0,crops,100799,0.086
1,animals,74676,0.062


**Answer.** Crop loans are under-funded more often than animal loans: 8.6% of the 100 799 crop loans did not reach their full amount, against 6.2% of the 74 676 animal loans. Most loans in both groups are fully funded, so the gap is only about 2.4 percentage points. I counted Aquaculture and Beekeeping as animals, and left out 4 827 loans from other activities such as Farm Supplies and Land Rental. The data cannot say why crops are less funded, only that they are.

# Question 6: Among funded Health loans, which activities (with enough loans to compare) have the longest median number of days between posted_time and funded_time, and how many loans is each based on?

In [8]:
# create days_to_fund, then select, group, sort
health = loans[loans['sector'] == 'Health'].copy()
health['days_to_fund'] = (health['funded_time'] - health['posted_time']).dt.total_seconds() / 86400

by_activity_days = (health[health['days_to_fund'].notna()]
                    .groupby('activity')
                    .agg(n_loans=('id', 'size'),
                         median_days=('days_to_fund', 'median'))
                    .sort_values('median_days', ascending=False)
                    .reset_index())
by_activity_days.round(1)

,activity,n_loans,median_days
0,Personal Medical Expenses,5010,10.1
1,Dental,278,8.6
2,Health,1183,7.7
3,Natural Medicines,808,6.6
4,Pharmacy,1009,6.5
5,Medical Clinic,321,5.6


**Answer.** Personal Medical Expenses loans take the longest to fund, with a median of 10.1 days across 5 010 loans, while Medical Clinic loans are fastest at 5.6 days (321 loans). Dental (8.6 days, 278 loans) and the "Health" activity (7.7 days, 1 183 loans) sit in the middle, and Natural Medicines and Pharmacy are about one day slower than Medical Clinic at 6.5 days. The 614 loans that were never funded are excluded, since they have no funding date, so these figures describe funded loans only.